# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to load, explore, and process the FAIR<sup>2</sup> dataset defined by a Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', None)}\n\n{getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets by `@id` and their fields (by `@id`)
print("Available record sets and their field @ids:")
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in this dataset metadata.")
else:
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
        print(f"- RecordSet @id: {rs_id}")
        fields = rs.get('field', []) if isinstance(rs, dict) else getattr(rs, 'field', [])
        if isinstance(fields, dict) and '@id' in fields:
            fields = [fields]  # Convert to list
        for field in fields:
            f_id = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', None)
            print(f"    - Field @id: {f_id}")

# As a shortcut, print a list of all available record set @ids
record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None) for rs in record_sets]
print("\nList of record set @ids:")
pprint.pprint(record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Find the main record set containing tabular data
# If there are multiple sets, select the most relevant (usually the largest/tabular one).

# Here, we need the @id for the main record set, which must be obtained from the previous cell's output.
main_record_set_id = None
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
    # A heuristic: choose the first non-None id
    if rs_id:
        main_record_set_id = rs_id
        break

if not main_record_set_id:
    raise RuntimeError("No record set @id found. Please check dataset metadata.")

# Extract data from each record set
dataframes = {}
for rs_id in record_set_ids:
    # Defensive, skip empty rs id
    if not rs_id:
        continue
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if not df.empty:
            dataframes[rs_id] = df
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

print("\nLoaded DataFrames for these record set @ids:")
for k in dataframes.keys():
    print(f"- {k}")

# Show columns of the main record set DataFrame
main_df = dataframes[main_record_set_id]
print("\nColumns in main DataFrame:")
pprint.pprint(list(main_df.columns))
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by key attributes.

In [ ]:
# Select a numeric field for analysis (find a likely numeric column by inspecting column names and dtypes)
numeric_cols = main_df.select_dtypes(include=['number']).columns.tolist()
if not numeric_cols:
    # Fallback: try to detect numeric-looking columns by name
    numeric_cols = [col for col in main_df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'metastasis' in col.lower()]
# Pick the first numeric candidate
numeric_field = numeric_cols[0] if numeric_cols else None

if not numeric_field:
    raise ValueError("No numeric field found for EDA.")

print(f"Using numeric field: {numeric_field}")

# Set an arbitrary threshold based on the field's range
threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 1
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Pick a group field (categorical) for demonstration (heuristic: a likely categorical field)
categorical_cols = main_df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
for col in categorical_cols:
    if main_df[col].nunique() < 10:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field}, showing mean of {numeric_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for demonstration.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field is defined, plot a boxplot
if group_field:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR² dataset loaded using the `mlcroissant` library. You reviewed its record sets, identified key fields by their `@id`, loaded the main tabular data, applied basic filtering, normalization, and grouping, and visualized the data distribution. This process provides a foundation for deeper domain-specific or machine learning analyses.

Be sure to refer back to the dataset schema (and the record set/field `@id`s) for precise documentation when building FAIR and reproducible machine learning workflows!